# Results Summary

Summarizes ablation study and serving performance results from W&B and serving CSVs.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

OUTPUT_DIR = Path('../outputs')

## Ablation Study

Pass@1, median solution execution time, and generation latency across base SLM, control SFT, and runtime-aware SFT.

In [2]:
ablation_raw = pd.read_csv(OUTPUT_DIR / 'wandb_ablation_runs_2026-04-20.csv')

MODEL_LABELS = {
    'jg/test/base_slm': 'Base SLM',
    'jg/test/control_sft': 'Control SFT',
    'jg/test/runtime_aware_sft': 'Runtime-Aware SFT',
}

ablation = (
    ablation_raw[ablation_raw['Name'].isin(MODEL_LABELS)]
    [['Name', 'pass_at_1', 'median_execution_time_s', 'avg_generation_latency_s', 'problems_with_passing']]
    .copy()
)
ablation['Model'] = ablation['Name'].map(MODEL_LABELS)
ablation['median_execution_time_ms'] = ablation['median_execution_time_s'] * 1000
ablation = ablation.set_index('Model').loc[['Base SLM', 'Control SFT', 'Runtime-Aware SFT']]
ablation = ablation[['pass_at_1', 'median_execution_time_ms', 'avg_generation_latency_s', 'problems_with_passing']]
ablation.columns = ['Pass@1', 'Median Exec Time (ms)', 'Avg Gen Latency (s)', 'Problems Passing']

print('Ablation Study Results')
print('=' * 70)
ablation

Ablation Study Results


,Pass@1,Median Exec Time (ms),Avg Gen Latency (s),Problems Passing
Model,,,,
Base SLM,0.3690,0.0890,0.8896,468
Control SFT,0.7090,0.0906,0.7528,751
Runtime-Aware SFT,0.7016,0.0897,0.7425,745


## Serving Performance

Throughput and GPU utilization for HuggingFace and vLLM backends.

In [3]:
serving_raw = pd.read_csv(OUTPUT_DIR / 'serving_full_results.csv')

MODEL_PATH_LABELS = {
    'Qwen/Qwen2.5-Coder-1.5B-Instruct': 'Base SLM',
    '/content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_merged': 'Runtime-Aware SFT',
}
BACKEND_LABELS = {'hf': 'HuggingFace', 'vllm': 'vLLM'}

serving = serving_raw.copy()
serving['Model'] = serving['model_path'].map(MODEL_PATH_LABELS)
serving['Backend'] = serving['backend'].map(BACKEND_LABELS)
serving = serving.dropna(subset=['Model'])
serving = serving.set_index(['Model', 'Backend'])
serving = serving[[
    'avg_latency_per_prompt_s',
    'throughput_prompts_per_s',
    'throughput_output_tokens_per_s',
    'avg_gpu_util_pct',
    'peak_cuda_memory_mb',
]]
serving.columns = [
    'Avg Latency/Prompt (s)',
    'Throughput (prompts/s)',
    'Throughput (tokens/s)',
    'Avg GPU Util (%)',
    'Peak CUDA Mem (MB)',
]

print('Serving Performance Results')
print('=' * 70)
serving

Serving Performance Results


Avg Latency/Prompt (s)  Throughput (prompts/s)  \
Model             Backend                                                       
Base SLM          HuggingFace                  0.6887                  1.4519   
                  vLLM                         0.0318                 31.4506   
Runtime-Aware SFT HuggingFace                  0.6695                  1.4936   
                  vLLM                         0.0317                 31.5716   

                               Throughput (tokens/s)  Avg GPU Util (%)  \
Model             Backend                                                
Base SLM          HuggingFace               233.5155           37.6229   
                  vLLM                     1781.6128           97.2131   
Runtime-Aware SFT HuggingFace               228.6202           37.2855   
                  vLLM                     1891.1069           96.7705   

                               Peak CUDA Mem (MB)  
Model             Backend                          
Base SLM          HuggingFace           3705.7910  
                  vLLM                 37602.0000  
Runtime-Aware SFT HuggingFace           3705.7910  
                  vLLM                 37614.0000

## Profiling Summary

Generation latency and peak memory from profiling runs (batch_size=64, A100).

In [4]:
profiled_raw = pd.read_csv(OUTPUT_DIR / 'wandb_profiled_runs_2026-04-20.csv')

PROFILED_LABELS = {
    'base_slm_profiled': 'Base SLM',
    'control_sft_profiled': 'Control SFT',
    'runtime_aware_sft_profiled': 'Runtime-Aware SFT',
}

profiled = (
    profiled_raw[profiled_raw['Name'].isin(PROFILED_LABELS)]
    [['Name', 'pass_at_1', 'avg_generation_latency_s', 'peak_cuda_memory_mb']]
    .copy()
)
profiled['Model'] = profiled['Name'].map(PROFILED_LABELS)
profiled = profiled.set_index('Model').loc[['Base SLM', 'Control SFT', 'Runtime-Aware SFT']]
profiled = profiled[['pass_at_1', 'avg_generation_latency_s', 'peak_cuda_memory_mb']]
profiled.columns = ['Pass@1', 'Avg Gen Latency (s)', 'Peak CUDA Mem (MB)']

print('Profiling Results (batch_size=64, A100)')
print('=' * 70)
profiled

Profiling Results (batch_size=64, A100)


,Pass@1,Avg Gen Latency (s),Peak CUDA Mem (MB)
Model,,,
Base SLM,0.3656,1.4390,6691.4780
Control SFT,0.7072,1.1274,6691.4780
Runtime-Aware SFT,0.6956,1.1731,6691.4780
